# Experiment 6 — PyDI LLMExtractor (LLM-only)

- **25 query rows** from Dataset 1 with missing values
- **5 attributes:** `bus_type`, `model_number`, `model`, `read_speed_mb_s`, `height_mm`
- **KB:** 100 rows sampled from datasets 2+3+4
- **Method:** PyDI `LLMExtractor` — no retrieval, LLM predicts from product description
- **Scoring:** partial match + 10% tolerance for numeric attributes

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI/venv/lib64/python3.12/site-packages')
sys.path.insert(0, '/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI/venv/lib/python3.12/site-packages')

sys.path.append('/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI')

In [ ]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import pandas as pd
import numpy as np
import random
import time
from langchain_ollama import ChatOllama
from PyDI.informationextraction.llm import LLMExtractor
from pydantic import BaseModel, Field
from typing import Optional

In [ ]:
random.seed(42)
np.random.seed(42)

In [ ]:
TARGET_ATTRIBUTES = ["bus_type", "model_number", "model", "read_speed_mb_s", "height_mm"]
NUMERIC_ATTRIBUTES = {"read_speed_mb_s", "height_mm"}

## 2. Load Datasets

In [ ]:
df1 = pd.read_json("normalized_products/dataset_1_normalized.json")
df2 = pd.read_json("normalized_products/dataset_2_normalized.json")
df3 = pd.read_json("normalized_products/dataset_3_normalized.json")
df4 = pd.read_json("normalized_products/dataset_4_normalized.json")

kb_full = pd.concat([df2, df3, df4], ignore_index=True)

## 3. Build Evaluation Set (25 rows)

In [ ]:
def get_ground_truth(cluster_id, attribute, kb):
    """Find ground truth via cluster_id match in full KB."""
    matches = kb_full[kb_full["cluster_id"] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attribute)
        if pd.notna(val) and str(val).strip().lower() not in {"", "none", "nan"}:
            return str(val).strip()
    return None

# Find 25 rows with at least one missing recoverable attribute
selected_rows = []
for idx, row in df1.iterrows():
    missing_attrs = []
    for attr in TARGET_ATTRIBUTES:
        if pd.isna(row.get(attr)):
            gt = get_ground_truth(row["cluster_id"], attr, kb_full)
            if gt is not None:
                missing_attrs.append((attr, gt))
    if missing_attrs:
        selected_rows.append({"df1_idx": idx, "missing_attrs": missing_attrs})
    if len(selected_rows) == 25:
        break

# Build flat eval dataframe
eval_records = []
for item in selected_rows:
    for attr, gt in item["missing_attrs"]:
        eval_records.append({
            "df1_idx": item["df1_idx"],
            "attribute": attr,
            "ground_truth": gt,
            "is_numeric": attr in NUMERIC_ATTRIBUTES
        })

eval_df = pd.DataFrame(eval_records)
query_indices = [item["df1_idx"] for item in selected_rows]
query_df = df1.loc[query_indices].copy()

print(f"Query rows: {len(query_df)}")
print(f"Total eval tasks: {len(eval_df)}")
print()
print("Tasks per attribute:")
print(eval_df["attribute"].value_counts())

In [ ]:
# Build KB: first include all rows that have ground truth for our missing values
# then fill up to 1000 with random rows

# Get cluster_ids of our 25 query rows
query_cluster_ids = query_df["cluster_id"].tolist()

# Get KB rows that match our query cluster_ids (these have the ground truth values)
relevant_kb = kb_full[kb_full["cluster_id"].isin(query_cluster_ids)].copy()
print(f"Relevant KB rows (with ground truth): {len(relevant_kb)}")

# Fill remaining up to 1000 with random rows not in relevant_kb
remaining_needed = max(0, 100 - len(relevant_kb))
irrelevant_kb = kb_full[~kb_full["cluster_id"].isin(query_cluster_ids)]
random_fill = irrelevant_kb.sample(n=min(remaining_needed, len(irrelevant_kb)), random_state=42)
print(f"Random fill rows: {len(random_fill)}")

# Combine
kb = pd.concat([relevant_kb, random_fill], ignore_index=True)
print(f"Final KB size: {len(kb)}")

In [ ]:
# Sanity check: for each query row + missing attribute, 
# verify the ground truth value exists in the KB

print("=== SANITY CHECK: Ground truth in KB ===\n")
found = 0
not_found = 0

for _, task in eval_df.iterrows():
    idx = task["df1_idx"]
    attr = task["attribute"]
    gt = task["ground_truth"]
    cluster_id = query_df.loc[idx, "cluster_id"]
    
    # Find matching rows in KB by cluster_id
    kb_matches = kb[kb["cluster_id"] == cluster_id]
    
    # Check if ground truth value is in KB
    val_in_kb = False
    for _, kb_row in kb_matches.iterrows():
        val = kb_row.get(attr)
        if pd.notna(val) and str(val).strip().lower() not in {"", "none", "nan"}:
            if str(val).strip() == str(gt).strip():
                val_in_kb = True
                break
    
    status = "✓" if val_in_kb else "✗"
    if val_in_kb:
        found += 1
    else:
        not_found += 1
    
    print(f"{status} Row {idx} | {attr:<20} | GT: {str(gt):<30} | KB matches: {len(kb_matches)}")

print(f"\nSummary: {found} found in KB, {not_found} NOT found in KB")
print(f"Coverage: {100*found/(found+not_found):.1f}%")


## 4. PyDI LLMExtractor Setup

In [ ]:
chat_model = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11435")

# Test connection
test = chat_model.invoke("Say OK")
print("Ollama OK:", repr(test.content[:30]))

class ProductAttributes(BaseModel):
    bus_type: Optional[str] = Field(
        None, description="Interface bus type e.g. PCIe 3.0, SATA III, USB 3.0")
    model_number: Optional[str] = Field(
        None, description="Exact product model number or SKU")
    model: Optional[str] = Field(
        None, description="Product model name")
    read_speed_mb_s: Optional[float] = Field(
        None, description="Sequential read speed in MB/s, number only")
    height_mm: Optional[float] = Field(
        None, description="Product height in millimeters, number only")

system_prompt = """You are a product data expert specializing in computer hardware.
Extract the following attributes from the product information.
For numeric attributes (read_speed_mb_s, height_mm), return only the number.
If an attribute cannot be determined from the text, return null.
Return valid JSON only, no explanation."""

extractor = LLMExtractor(
    chat_model=chat_model,
    source_column="title_description",
    system_prompt=system_prompt,
    schema=ProductAttributes,
)
print("LLMExtractor ready.")

## 5. Run Extraction

In [ ]:
print(f"Running PyDI LLMExtractor on {len(query_df)} rows...")
t0 = time.time()

In [ ]:
extracted_df = extractor.extract(query_df)

In [ ]:
print(f"Done in {time.time()-t0:.1f}s")
print()
print("Sample extracted values:")
print(extracted_df[TARGET_ATTRIBUTES].to_string())

## 6. Score Results

In [ ]:
def normalize(val):
    return str(val).lower().strip()

def is_correct(predicted, ground_truth, attribute):
    if predicted is None or str(predicted).strip().lower() in {
            "", "nan", "none", "unknown", "null"}:
        return False
    if attribute in NUMERIC_ATTRIBUTES:
        try:
            p = float(str(predicted).replace(",", "").strip())
            g = float(str(ground_truth).replace(",", "").strip())
            return abs(p - g) / abs(g) <= 0.10 if g != 0 else p == 0
        except:
            pass
    p = normalize(str(predicted))
    g = normalize(str(ground_truth))
    return p == g or p in g or g in p

results = []
for _, task in eval_df.iterrows():
    idx = task["df1_idx"]
    attr = task["attribute"]
    gt = task["ground_truth"]

    if attr in extracted_df.columns and idx in extracted_df.index:
        predicted = extracted_df.loc[idx, attr]
        predicted_str = str(predicted) if pd.notna(predicted) and predicted is not None else "UNKNOWN"
    else:
        predicted_str = "UNKNOWN"

    correct = is_correct(predicted_str, gt, attr)
    unknown = predicted_str.upper() == "UNKNOWN" or predicted_str.lower() in {"nan", "none", "null"}

    results.append({
        "config": "PyDI-LLMExtractor",
        "df1_idx": idx,
        "attribute": attr,
        "is_numeric": task["is_numeric"],
        "ground_truth": gt,
        "predicted": predicted_str,
        "correct": correct,
        "unknown": unknown
    })

results_df = pd.DataFrame(results)
results_df.to_csv("exp6_results_llm.csv", index=False)
print("✓ Saved to exp6_results_llm.csv")

## 7. Results

In [ ]:
print("=" * 55)
print("RESULTS — PyDI LLMExtractor (LLM-only, no KB)")
print("=" * 55)
print(f"Overall accuracy: {results_df['correct'].mean():.3f}")
print(f"UNKNOWN rate:     {results_df['unknown'].mean():.3f}")
print(f"Total tasks:      {len(results_df)}")
print()
print("Per-attribute breakdown:")
summary = results_df.groupby("attribute").agg(
    total=("correct", "count"),
    correct=("correct", "sum"),
    accuracy=("correct", "mean"),
    unknown_rate=("unknown", "mean")
).round(3)
print(summary.to_string())
print()
print("Sample predictions:")
print(results_df[["attribute", "ground_truth", "predicted", "correct"]]
      .head(15).to_string(index=False))

In [ ]:
def build_few_shot_prompt(text):
    return f"""You are a product data expert. Extract attributes from product text.
The model_number is usually a specific alphanumeric code found in the title or parentheses.
The bus_type describes the interface (PCIe, SATA, USB).

--- EXAMPLE 1 ---
Product: WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM SATA 6Gb/s 256MB Cache 3.5 Inch - WD60EZAZ. Description: WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM SATA 6Gb/s 256MB Cache 3.5 Inch - WD60EZAZ
Output: {{"bus_type": "SATA III", "model_number": "WD60EZAZ", "model": "WD Blue", "read_speed_mb_s": null, "height_mm": null}}

--- EXAMPLE 2 ---
Product: CORSAIR - Force Series MP510 960GB M.2 SSD PCIe Gen3 x4, M.2 NVMe, up to 3480/3000MB/s read/write. Description: CORSAIR Force Series MP510 960GB M.2 SSD (CSSD-F960GBMP510) - PCI Express 3.0 - 960GB
Output: {{"bus_type": "PCIe 3.0 x4", "model_number": "CSSD-F960GBMP510", "model": "Force Series MP510", "read_speed_mb_s": 3480.0, "height_mm": null}}

--- EXAMPLE 3 ---
Product: 4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/s, 512e, 128MB cache, 3.5-Inch HDD. Description: 4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/s, 512e, 128MB cache, 3.5-Inch HDD
Output: {{"bus_type": "SATA III", "model_number": "ST4000NM0115", "model": "Exos 7E8", "read_speed_mb_s": null, "height_mm": null}}

--- EXAMPLE 4 ---
Product: XPG GAMMIX 1TB S11 Pro 3D NAND PCIe NVMe Gen3x4 M.2 2280 SSD (AGAMMIXS11P-1TT-C). Description: PCIe Gen3x4 interface, NVMe 1.3, read/write speeds up to 3500/3000MB per second.
Output: {{"bus_type": "PCIe 3.0 x4", "model_number": "AGAMMIXS11P-1TT-C", "model": "GAMMIX S11 Pro", "read_speed_mb_s": 3500.0, "height_mm": null}}

--- EXAMPLE 5 ---
Product: Gigabyte AMD Radeon RX 5600 XT GAMING OC Video Card GV-R56XTGAMING OC-6GD. Description: Interface: PCI-E 4.0 x 16, 6GB GDDR6, Core Clock: 1620 MHz
Output: {{"bus_type": "PCIe 4.0 x16", "model_number": "GV-R56XTGAMING OC-6GD", "model": "AMD Radeon RX 5600 XT GAMING OC", "read_speed_mb_s": null, "height_mm": null}}

--- NOW EXTRACT ---
Product: {str(text)[:600]}
Output:"""

In [ ]:
def llm_evaluate(predicted, ground_truth, attribute, chat_model):
    if predicted == "UNKNOWN" or str(predicted).lower() in {"nan", "none", "null"}:
        return "wrong"
    
    eval_prompt = f"""You are evaluating a data cleaning prediction for a product database.

Attribute: {attribute}
Ground truth value: {ground_truth}
Predicted value: {predicted}

Judge the prediction as exactly one of:
- CORRECT: exact match or semantically equivalent (e.g. "PCIe Gen3x4" = "PCIe 3.0 x4")
- ACCEPTABLE: more/less detailed but not wrong (e.g. "PCIe 3.0" when GT is "PCIe 3.0 x16")
- WRONG: clearly incorrect value

Respond with JUDGMENT:<label> only. Example: JUDGMENT:CORRECT"""

    response = chat_model.invoke([HumanMessage(content=eval_prompt)])
    response_text = response.content.strip().upper()
    
    if "JUDGMENT:CORRECT" in response_text:
        return "correct"
    elif "JUDGMENT:ACCEPTABLE" in response_text:
        return "acceptable"
    else:
        return "wrong"

In [ ]:
print("Running LLM-based evaluation...")
llm_judgments = []
for i, row in results_df.iterrows():
    judgment = llm_evaluate(
        row["predicted"], row["ground_truth"], 
        row["attribute"], chat_model
    )
    llm_judgments.append(judgment)
    print(f"  {row['attribute']:<20} | GT: {str(row['ground_truth']):<25} | "
          f"Pred: {str(row['predicted']):<25} | {judgment}")

results_df["llm_judgment"] = llm_judgments
results_df.to_csv("exp6_results_llm.csv", index=False)

print("\n" + "="*55)
print("COMPARISON: String Match vs LLM Evaluation")
print("="*55)

# String match accuracy
string_acc = results_df["correct"].mean()

# LLM accuracy (correct only)
llm_correct = (results_df["llm_judgment"] == "correct").mean()

# LLM accuracy (correct + acceptable)
llm_acceptable = (results_df["llm_judgment"].isin(["correct", "acceptable"])).mean()

print(f"String match accuracy:          {string_acc:.3f}")
print(f"LLM evaluation (correct only):  {llm_correct:.3f}")
print(f"LLM evaluation (correct+acceptable): {llm_acceptable:.3f}")

print("\nPer-attribute comparison:")
per_attr = results_df.groupby("attribute").apply(lambda x: pd.Series({
    "string_match": x["correct"].mean(),
    "llm_correct": (x["llm_judgment"] == "correct").mean(),
    "llm_acceptable": (x["llm_judgment"].isin(["correct", "acceptable"])).mean(),
    "total": len(x)
})).round(3)
print(per_attr.to_string())

print("\nLLM judgment distribution:")
print(results_df["llm_judgment"].value_counts())

In [ ]:
# Check what prompt PyDI is actually building
extractor2 = LLMExtractor(
    chat_model=chat_model,
    source_column="title_description",
    system_prompt=system_prompt,
    schema=ProductAttributes,
    debug=True  # enable debug to see prompts
)

# Test on just 1 row
test_row = query_df.head(1)
print("Input text:", test_row["title_description"].iloc[0][:200])
print()

# Build the prompt template and show it
pt = extractor2._build_prompt_template()
msgs = pt.format_messages(text=test_row["title_description"].iloc[0])
for m in msgs:
    print(f"=== {m.type.upper()} ===")
    print(m.content[:500])
    print()

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

class SimpleLLMExtractor:
    """Simple LLM extractor that bypasses PyDI's broken schema formatting."""
    
    def __init__(self, chat_model):
        self.chat_model = chat_model
    
    def extract(self, df, source_column="title_description"):
        results = []
        for idx, row in df.iterrows():
            text = row.get(source_column, "")
            if not text or pd.isna(text):
                results.append({"df1_idx": idx, 
                                 "bus_type": None, "model_number": None,
                                 "model": None, "read_speed_mb_s": None, 
                                 "height_mm": None})
                continue
            
            prompt = f"""Extract product attributes from the text below. Return ONLY a JSON object.

Product text: {str(text)[:500]}

Extract these fields:
- "bus_type": interface type found in text (e.g. "PCIe 3.0 x16", "SATA III", "USB 3.0", "NVMe", "PCI Express 3.0 x4")
- "model_number": manufacturer part number / SKU (e.g. "GV-N3080GAMING OC-10GD", "MZ-V7E500BW") — usually alphanumeric with dashes
- "model": product model name (e.g. "Force MP510", "Gaming OC", "970 EVO")
- "read_speed_mb_s": read speed number only in MB/s (e.g. 3500) — only if explicitly mentioned
- "height_mm": height in mm number only (e.g. 46.0) — only if explicitly mentioned

Set field to null if not found. Do not guess.

JSON:"""

            response = self.chat_model.invoke([HumanMessage(content=prompt)])
            text_response = response.content.strip()
            
            # Parse JSON
            import json, re
            try:
                # Strip code fences if present
                match = re.search(r'\{.*\}', text_response, re.DOTALL)
                if match:
                    data = json.loads(match.group(0))
                else:
                    data = {}
            except:
                data = {}
            
            results.append({
                "df1_idx": idx,
                "bus_type": data.get("bus_type"),
                "model_number": data.get("model_number"),
                "model": data.get("model"),
                "read_speed_mb_s": data.get("read_speed_mb_s"),
                "height_mm": data.get("height_mm")
            })
            print(f"  Row {idx}: {data}")
        
        result_df = pd.DataFrame(results).set_index("df1_idx")
        return result_df

# Test on 2 rows first
simple_extractor = SimpleLLMExtractor(chat_model)
test_result = simple_extractor.extract(query_df.head(2))
print(test_result)

In [ ]:
print(query_df.loc[2, "title_description"][:500])